In [0]:
%sql
-- 1. Separando os filmes em categorias de sucesso na internet
WITH EngajamentoInternet AS (
    SELECT 
        TITULO_ORIGINAL,
        PUBLICO,
        ORCAMENTO_USD,
        yt_views,
        yt_sentiment,
        rotten_tomatoes,
        -- Criando uma régua de negócio baseada na nota da crítica
        CASE 
            WHEN rotten_tomatoes >= 80 THEN '1. Aclamado pela Crítica (>80%)'
            WHEN rotten_tomatoes BETWEEN 50 AND 79 THEN '2. Mediano (50-79%)'
            WHEN rotten_tomatoes > 0 AND rotten_tomatoes < 50 THEN '3. Bombardeado (<50%)'
            ELSE '4. Sem Avaliação'
        END AS CATEGORIA_CRITICA
    FROM workspace.default.base_kinoplex_ml_binarizada
    -- Filtrando apenas filmes que tiveram alguma visualização no YT pra não sujar a média
    WHERE yt_views > 0
)

-- 2. Agregando o resultado final para o Painel Gerencial
SELECT 
    CATEGORIA_CRITICA,
    COUNT(TITULO_ORIGINAL) AS QUANTIDADE_DE_FILMES,
    ROUND(AVG(PUBLICO), 0) AS MEDIA_DE_PUBLICO_POR_FILME,
    ROUND(AVG(yt_sentiment), 2) AS MEDIA_SENTIMENTO_YOUTUBE,
    ROUND(AVG(ORCAMENTO_USD), 0) AS MEDIA_ORCAMENTO_USD
FROM EngajamentoInternet
GROUP BY CATEGORIA_CRITICA
ORDER BY CATEGORIA_CRITICA;